In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import signal
from tqdm import tqdm

In [2]:

def outlier_remover(x, z = 4):
    
    xc = x.replace(9999,np.nan) # remove blinks
    xz = (xc - xc.mean())/xc.std() # z-score
    xz[xz > z] = np.nan # remove values above 3 z score
    xz[xz < -z] = np.nan # remove values below -3 z score
    xz = xz.fillna(method='ffill') # change nans to previous value
    xz = xz.fillna(method = 'backfill') # change any nans at the beginning 
    
    return xz

def emp_outlier_remover(x, z = 4):
    
    xc = x.replace(9999,np.nan) # remove blinks
    xz = (xc - xc.mean())/xc.std() # z-score
    xc[xz > z] = np.nan # remove values above 3 z score
    xc[xz < -z] = np.nan # remove values below -3 z score
    xc = xc.fillna(method='ffill') # change nans to previous value
    xc = xc.fillna(method = 'backfill') # change any nans at the beginning 
    
    return xc

def corrmean(data):
    return np.tanh(np.nanmean(np.arctanh(data)))

def bootstrap_mean_corr(data,lo_ci = 2.5, hi_ci = 97.5,reps = 5000):
    
    ''' Computes bootstrapped mean and 95% confidence intervals with fisher z trasformation for correlations'''
    n = len(data)
    data = np.array((data))
    
    boot_data = [data[np.random.choice(range(n), size=n, replace=True)] for x in range(reps)] # get n datasets
    means = [np.tanh(np.nanmean(np.arctanh(x))) for x in boot_data] # get mean of each dataset
    
    m = np.tanh(np.nanmean(np.arctanh(means))) # fisher z transform
    ci = np.nanpercentile(means, [lo_ci, hi_ci]) # confidence intervals
    
    return m, ci

def bootstrap_mean(data,lo_ci = 2.5, hi_ci = 97.5,reps = 5000):
    
    ''' Computes bootstrapped mean and 95% confidence intervals with fisher z trasformation for correlations'''
    n = len(data)
    data = np.array((data))
    
    boot_data = [data[np.random.choice(range(n), size=n, replace=True)] for x in range(reps)] # get n datasets
    means = [np.nanmean(x) for x in boot_data] # get mean of each dataset
    
    m = np.nanmean(means)  
    ci = np.nanpercentile(means, [lo_ci, hi_ci]) # confidence intervals
    
    return m, ci

In [3]:
import cv2
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from vit_pytorch import ViT

def get_cv_frames(file_path):
    """Gets frame count using OpenCV and ensures resource release."""
    cap = None
    try:
        cap = cv2.VideoCapture(str(file_path))
        if not cap.isOpened():
            return "Error: Could not open"
        # Get the frame count property
        return int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    except Exception as e:
        return f"Error: {e}"
    finally:
        if cap:
            cap.release()

# --- Configuration ---
video_folder = Path("/home/Jeff/1.0 projects/intersubject gaze correlations/videos")
video_extensions = {'.mp4', '.avi', '.mov', '.mkv'} # Use a set for fast lookup
# ---------------------

print(f"Scanning: {video_folder}")

# Use a dictionary comprehension to build the result
frame_counts = {
    p.name: get_cv_frames(p)
    for p in video_folder.glob('*')
    if p.is_file() and p.suffix.lower() in video_extensions
}

import re
# Create a new, clean dictionary
renamed_frame_counts = {}

for filename, frames in frame_counts.items():
    # Find all sequences of digits
    numbers_found = re.findall(r'\d+', filename[:-4])
    
    # Get the *last* number found as a string
    video_number_str = numbers_found[-1]
    renamed_frame_counts[video_number_str] = frames
# Print the new dictionary
renamed_frame_counts

Scanning: /home/Jeff/1.0 projects/intersubject gaze correlations/videos


{'032': 1500,
 '005': 1681,
 '023': 1500,
 '011': 2064,
 '031': 1500,
 '020': 1674,
 '040': 1500,
 '004': 2369}

In [4]:
videos = {}
ratings = {}

dataDir = '/home/Jeff/1.0 projects/intersubject gaze correlations/data/experiment 2/'
os.chdir(dataDir)

context_rating_data = {}
context_gaze_data = {}

# iterate through folders
## gaze data
for folder in os.listdir():
    
    os.chdir(dataDir + folder)

    # identify and store core files
    gazeFile = [x for x in os.listdir() if 'gaze' in x][0]
    ratingFile = folder + '.csv'

    ## gaze data
    df = outlier_remover(pd.read_csv(gazeFile).iloc[:,1:].dropna())

    # create new dataframe for column
    for column in df.columns:
            
        # store data

        if column not in context_gaze_data:
            context_gaze_data[column] = pd.DataFrame()
        r = signal.resample(df[column], renamed_frame_counts[column.split('_')[0]])
        context_gaze_data[column] = pd.concat([context_gaze_data[column],pd.Series(r).rename(folder)], axis = 1)

    ## rating data
    df = outlier_remover(pd.read_csv(ratingFile).iloc[:,1:].dropna())

    # create new dataframe for column
    for column in df.columns:
        if column not in context_rating_data:
            context_rating_data[column] = pd.DataFrame()

        # store data
        if column not in context_rating_data:
            context_rating_data[column] = pd.DataFrame()
        r = signal.resample(df[column], renamed_frame_counts[column.split('_')[0]])
        context_rating_data[column] = pd.concat([context_rating_data[column],pd.Series(r).rename(folder)], axis = 1)
        
    os.chdir(dataDir)


In [5]:
context_rating_mean = {s:[context_rating_data[s].mean(axis = 1)] for s in context_rating_data}
context_rating_mean

valence_mean = {s.split('_')[0]:context_rating_mean[s][0] for s in context_rating_mean if 'valence' in s}

In [8]:
import os
import re
import glob
import cv2  # This is the OpenCV library

class VideoFrameDataset(Dataset):
    """
    This dataset loads frames from video files and matches them with
    ratings from a provided dictionary.
    """
    def __init__(self, video_dir, ratings_dict, image_size=256):
        self.image_size = image_size
        self.frame_map = [] # This will store (video_path, frame_index, rating)

        print("Scanning video files and building frame map...")
        video_paths = glob.glob(os.path.join(video_dir, "*.mp4"))
        
        for video_path in video_paths:
            video_name = os.path.basename(video_path)
            
            # --- 1. Extract the key (e.g., '011') ---
            # This regex finds the first sequence of digits in the filename.
            match = re.search(r"(\d{3})", video_name)
            
            if not match:
                print(f"Warning: Could not find number key in {video_name}. Skipping.")
                continue
                
            key = match.group(1)
            
            # --- 2. Check if key exists in your ratings ---
            if key in ratings_dict:
                ratings_list = ratings_dict[key]
                
                # --- 3. Open video to get frame count ---
                try:
                    cap = cv2.VideoCapture(video_path)
                    num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                    
                    # --- 4. Check for length mismatch ---
                    if num_frames != len(ratings_list):
                        print(f"Warning: Mismatch in {video_name}. "
                              f"Video has {num_frames} frames, but {len(ratings_list)} ratings. Skipping.")
                        cap.release()
                        continue
                        
                    # --- 5. Create the map ---
                    for frame_idx in range(num_frames):
                        rating = ratings_list[frame_idx]
                        self.frame_map.append((video_path, frame_idx, rating))
                        
                    cap.release()
                    
                except Exception as e:
                    print(f"Error processing video {video_name}: {e}")

            else:
                print(f"Warning: Key '{key}' from {video_name} not found in ratings dictionary. Skipping.")
                
        print(f"Done. Found {len(self.frame_map)} total frames.")

    def __len__(self):
        # Total number of frames in the dataset
        return len(self.frame_map)

    def __getitem__(self, idx):
        # 1. Get the info for this frame
        video_path, frame_idx, rating = self.frame_map[idx]
        
        # 2. Open the video file
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise IOError(f"Failed to open video file: {video_path}")
        
        # 3. Go to the specific frame
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        
        # 4. Read the frame
        ret, frame = cap.read()
        cap.release()
        
        if not ret:
            raise RuntimeError(f"Failed to read frame {frame_idx} from {video_path}")
            
        # 5. Process the frame (convert to RGB, resize, to-tensor)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        frame_resized = cv2.resize(frame_rgb, (self.image_size, self.image_size), 
                                   interpolation=cv2.INTER_AREA)
        
        # Convert HWC (Height, Width, Channel) to CHW (Channel, Height, Width)
        # and normalize to [0, 1]
        frame_tensor = torch.tensor(frame_resized).permute(2, 0, 1) / 255.0
        
        # Ensure rating is a float tensor with the right shape
        rating_tensor = torch.tensor(rating).float().view(1)
        
        return frame_tensor, rating_tensor
    

In [9]:
# Check if a GPU (like NVIDIA) is available, otherwise use the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# 1.0 training model on 1 video

In [24]:
# --- 3. Model, Loss, and Optimizer Setup ---
# ... (all your imports and the Dataset class definition go above this) ...


VIDEO_DIRECTORY = "/home/Jeff/1.0 projects/intersubject gaze correlations/data/video training" # <-- SET THIS

# Create an instance of the dataset and the DataLoader
BATCH_SIZE = 128
train_dataset = VideoFrameDataset(video_dir=VIDEO_DIRECTORY, 
                                  ratings_dict=valence_mean, 
                                  image_size=256)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# ... (The rest of your code: model init, loss, optimizer, training loop)
# ... (This part does not need to change at all)


# Initialize the ViT to output a single continuous value
model = ViT(
    image_size = 256,
    patch_size = 32,
    num_classes = 1,  # Output one value (your valence rating)
    dim = 1024,
    depth = 6,
    heads = 16,
    mlp_dim = 2048
).to(device) # <-- Move the model to the GPU

# Mean Squared Error Loss for regression
loss_function = nn.MSELoss()

# Adam optimizer to update the model's weights
optimizer = optim.Adam(model.parameters(), lr=1e-5) # 1e-5 is a common, small learning rate

# --- 4. The Training Loop ---

NUM_EPOCHS = 10 # An epoch is one full pass through the entire dataset
loss_couter = []

print("--- Starting Training ---")
model.train() # Set the model to training mode

for epoch in tqdm(range(NUM_EPOCHS)):
    total_loss = 0.0
    
    # store values
    model_pred = []
    raw_data = []
    
    # Loop over each batch of data from the DataLoader
    for batch_idx, (frames, ratings) in enumerate(train_loader):
        
        # --- Move data to the same device as the model (GPU/CPU) ---
        frames = frames.to(device)
        ratings = ratings.to(device).float() # Ensure ratings are float
        
        # --- THE 5 KEY TRAINING STEPS ---
        
        # 1. Clear old gradients
        optimizer.zero_grad()

        # 2. Make a prediction (Forward Pass)
        predicted_valence = model(frames)
        model_pred.append(predicted_valence)
        raw_data.append(ratings)
        
        # 3. Calculate the error (Loss)
        loss = loss_function(predicted_valence, ratings)

        # 4. Calculate adjustments (Backward Pass), 
        loss.backward()

        # 5. Apply adjustments (Optimizer Step)
        optimizer.step()
        
        total_loss += loss.item() # Add the loss for this batch to the total
    
    # Print a summary for this epoch
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} --- Average Loss: {avg_loss:.6f}")
    loss_couter.append(avg_loss)
print("--- Training Finished ---")

# After training, you can save your model
# torch.save(model.state_dict(), 'valence_model.pth')

Scanning video files and building frame map...
Done. Found 1500 total frames.
--- Starting Training ---


 10%|█         | 1/10 [03:33<31:57, 213.01s/it]

Epoch 1/10 --- Average Loss: 0.626289


 20%|██        | 2/10 [07:06<28:26, 213.30s/it]

Epoch 2/10 --- Average Loss: 0.319219


...

Epoch 8/10 --- Average Loss: 0.016043


 90%|█████████ | 9/10 [32:02<03:33, 213.66s/it]

Epoch 9/10 --- Average Loss: 0.014645


100%|██████████| 10/10 [35:36<00:00, 213.62s/it]

Epoch 10/10 --- Average Loss: 0.014127
--- Training Finished ---


# train on all videos and test on left out video

In [ ]:
import os
import re
import glob
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from pathlib import Path
from scipy.stats import pearsonr
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from vit_pytorch import ViT

# --- 1. Configuration ---
VIDEO_DIRECTORY = "/home/Jeff/1.0 projects/intersubject gaze correlations/videos" # <-- SET THIS
BATCH_SIZE = 128
NUM_EPOCHS = 10 # Epochs to train *per fold*
LEARNING_RATE = 1e-5
IMAGE_SIZE = 256
PATCH_SIZE = 32

# Set the device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- !!! IMPORTANT: MOCK DATA !!! ---
# You MUST replace this with your actual loaded 'valence_mean' dictionary.
# This mock data uses the structure you defined (key -> list of ratings).
# Your filenames (e.g., 'video_001.mp4', 'data_002.mp4') MUST
# contain the key (e.g., '001', '002') for the regex to find.
try:
    # This will fail, so the 'except' block will run.
    # Replace this 'try/except' block with the code that loads your real data.
    # Example: valence_mean = load_my_ratings_pickle('ratings.pkl')
    _ = valence_mean
    print("Loaded existing 'valence_mean' variable.")
except NameError:
    print("WARNING: 'valence_mean' not found. Using mock data.")
    print("         Replace the 'try/except' block with your actual data loading code.")
    valence_mean = {
        '001': [0.5, 0.6, 0.4] * 10,  # Mock 30 frames for video '001'
        '002': [-0.1, -0.2, -0.1] * 10, # Mock 30 frames for video '002'
        '003': [0.8, 0.7, 0.9, 0.8] * 7, # Mock 28 frames for video '003'
    }
    print("Mock data expects video files with names like 'vid_001.mp4', 'vid_002.mp4', etc.")
# --- End of Mock Data Section ---


# --- 2. Your VideoFrameDataset Class (with minor edits for LOOCV) ---
class VideoFrameDataset(Dataset):
    """
    This dataset loads frames from video files and matches them with
    ratings from a provided dictionary.
    
    -- MODIFIED FOR LOOCV --
    """
    def __init__(self, video_paths_list, ratings_dict, image_size=256):
        """
        MODIFICATION 1: Changed `video_dir` to `video_paths_list`.
        This allows us to pass a specific list of videos (e.g., N-1 for training
        or 1 for testing) instead of scanning a whole directory.
        """
        self.image_size = image_size
        self.frame_map = [] # This will store (video_path, frame_index, rating)

        # Define the image transformations for ViT
        # This is standard for models pretrained on ImageNet
        self.transforms = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        print(f"Scanning {len(video_paths_list)} video file(s) and building frame map...")
        
        # Iterate over the provided list of Path objects
        for video_path in video_paths_list:
            video_name = video_path.name
            
            # --- 1. Extract the key (e.g., '011') ---
            match = re.search(r"(\d{3})", video_name)
            
            if not match:
                print(f"Warning: Could not find number key in {video_name}. Skipping.")
                continue
                
            key = match.group(1)
            
            # --- 2. Check if key exists in your ratings ---
            if key in ratings_dict:
                ratings_list = ratings_dict[key]
                
                # --- 3. Open video to get frame count ---
                try:
                    cap = cv2.VideoCapture(str(video_path))
                    if not cap.isOpened():
                         print(f"Warning: Could not open {video_path}. Skipping.")
                         continue
                         
                    num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                    
                    # --- 4. Check for length mismatch ---
                    if num_frames != len(ratings_list):
                        print(f"Warning: Mismatch in {video_name}. "
                              f"Video has {num_frames} frames, but {len(ratings_list)} ratings. Skipping.")
                        cap.release()
                        continue
                        
                    # --- 5. Create the map ---
                    for frame_idx in range(num_frames):
                        rating = ratings_list[frame_idx]
                        self.frame_map.append((str(video_path), frame_idx, rating))
                        
                    cap.release()
                    
                except Exception as e:
                    print(f"Error processing video {video_name}: {e}")

            else:
                print(f"Warning: Key '{key}' from {video_name} not found in ratings dictionary. Skipping.")
                
        print(f"Done. Found {len(self.frame_map)} total frames for this dataset.")
        if not self.frame_map:
             print("Warning: No frames were loaded. Check video paths and ratings keys.")

    def __len__(self):
        return len(self.frame_map)

    def __getitem__(self, idx):
        video_path, frame_idx, rating = self.frame_map[idx]
        
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise IOError(f"Failed to open video file: {video_path}")
        
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        
        ret, frame = cap.read()
        cap.release()
        
        if not ret:
            raise RuntimeError(f"Failed to read frame {frame_idx} from {video_path}")
            
        # --- MODIFICATION 2: Use self.transforms ---
        # Convert BGR (OpenCV default) to RGB (PIL/PyTorch default)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Apply the defined transforms (Resize, ToTensor, Normalize)
        frame_tensor = self.transforms(frame_rgb)
        
        # Ensure rating is a float tensor with the right shape
        rating_tensor = torch.tensor(rating).float().view(1) # [1]
        
        return frame_tensor, rating_tensor

# --- 3. Metrics Calculation Function ---
def calculate_metrics(y_true, y_pred):
    """Calculates MSE, Pearson r, and Concordance Correlation Coefficient (CCC)."""
    
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    
    if len(y_true) < 2:
        print("Warning: Need at least 2 samples to calculate correlation. Returning NaNs.")
        return np.nan, np.nan, np.nan

    # Mean Squared Error
    mse = np.mean((y_true - y_pred)**2)
    
    # Pearson Correlation
    # Handle case where all predictions or truths are constant
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        pearson_r, p_value = np.nan, np.nan
    else:
        pearson_r, p_value = pearsonr(y_true, y_pred)
    
    # Concordance Correlation Coefficient (CCC)
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)
    covar = np.cov(y_true, y_pred)[0, 1]
    
    # Handle potential division by zero if variance is zero
    denominator = (var_true + var_pred + (mean_true - mean_pred)**2)
    if denominator == 0:
        ccc = np.nan # Or 1.0 if y_true == y_pred
    else:
        ccc = (2 * covar) / denominator
    
    return mse, pearson_r, ccc

# --- 4. The Leave-One-Out Cross-Validation (LOOCV) Loop ---
def run_loocv():
    video_dir = Path(VIDEO_DIRECTORY)
    if not video_dir.exists():
        print(f"Error: VIDEO_DIRECTORY not found at: {VIDEO_DIRECTORY}")
        return

    # Get all video files
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv']
    all_video_paths = []
    for ext in video_extensions:
        all_video_paths.extend(list(video_dir.glob(f'*{ext}')))
        
    if not all_video_paths:
        print(f"Error: No video files found in {VIDEO_DIRECTORY}")
        return

    print(f"Found {len(all_video_paths)} videos. Starting LOOCV...")
    
    results = {} # To store the metrics for each fold

    # --- Start the main loop ---
    for i, test_video_path in enumerate(all_video_paths):
        fold_num = i + 1
        print(f"\n{'='*50}")
        print(f"LOOCV FOLD {fold_num}/{len(all_video_paths)}: Testing on {test_video_path.name}")
        print(f"{'='*50}")

        # 1. Split data into training and test sets
        train_video_paths = [p for p in all_video_paths if p != test_video_path]
        
        if not train_video_paths:
            print("Only one video found. Cannot perform LOOCV. Skipping.")
            continue

        # 2. Create Datasets and DataLoaders
        print("Initializing Training Dataset...")
        train_dataset = VideoFrameDataset(video_paths_list=train_video_paths, 
                                          ratings_dict=valence_mean, 
                                          image_size=IMAGE_SIZE)
        # Add a check for empty training dataset
        if len(train_dataset) == 0:
            print(f"Warning: Fold {fold_num} has 0 training samples. Skipping fold.")
            continue
            
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
        
        print("Initializing Test Dataset...")
        test_dataset = VideoFrameDataset(video_paths_list=[test_video_path], 
                                         ratings_dict=valence_mean, 
                                         image_size=IMAGE_SIZE)
        # Add a check for empty test dataset
        if len(test_dataset) == 0:
            print(f"Warning: Test video {test_video_path.name} has 0 samples. Skipping fold.")
            continue

        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

        # 3. Initialize Model, Loss, and Optimizer (CRITICAL: must be new for each fold)
        model = ViT(
            image_size=IMAGE_SIZE,
            patch_size=PATCH_SIZE,
            num_classes=1,  # Output one value (your valence rating)
            dim=1024,
            depth=6,
            heads=16,
            mlp_dim=2048
        ).to(device)

        loss_function = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

        # 4. The Training Loop (for this fold)
        print(f"--- Starting Training for fold {fold_num} ---")
        for epoch in range(NUM_EPOCHS):
            model.train() # Set model to training mode
            total_loss = 0.0
            
            pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False)
            for batch_idx, (frames, ratings) in enumerate(pbar):
                frames = frames.to(device)
                # Your Dataset already returns [B, 1], so no .unsqueeze(1) needed
                ratings = ratings.to(device).float() 
                
                optimizer.zero_grad()
                predicted_valence = model(frames)
                loss = loss_function(predicted_valence, ratings)
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
                pbar.set_postfix(loss=f"{loss.item():.4f}")
            
            avg_loss = total_loss / len(train_loader)
            print(f"Fold {fold_num} | Epoch {epoch+1}/{NUM_EPOCHS} --- Average Train Loss: {avg_loss:.6f}")
        
        print(f"--- Training Finished for fold {fold_num} ---")

        # 5. The Evaluation Loop (for this fold)
        print(f"--- Evaluating on {test_video_path.name} ---")
        model.eval() # Set model to evaluation mode
        
        all_predictions = []
        all_ground_truth = []
        
        with torch.no_grad(): # Disable gradient calculations
            for frames, ratings in test_loader:
                frames = frames.to(device)
                ratings = ratings.to(device).float() # Shape is [B, 1]
                
                predicted_valence = model(frames)
                
                all_predictions.extend(predicted_valence.cpu().numpy().flatten())
                all_ground_truth.extend(ratings.cpu().numpy().flatten())
        
        # 6. Calculate Metrics and Store Results
        mse, r, ccc = calculate_metrics(all_ground_truth, all_predictions)
        
        print(f"\n--- Results for {test_video_path.name} ---")
        print(f"  Mean Squared Error (MSE): {mse:.4f}")
        print(f"  Pearson Correlation (r):  {r:.4f}")
        print(f"  Concordance Corr (CCC):   {ccc:.4f}\n")
        
        results[test_video_path.name] = {'mse': mse, 'pearson_r': r, 'ccc': ccc}

    # --- 7. Final Report ---
    print(f"\n{'='*50}")
    print("--- All LOOCV Folds Complete ---")
    print(f"{'='*50}")
    
    avg_mse = []
    avg_r = []
    avg_ccc = []
    
    for video_name, metrics in results.items():
        print(f"{video_name}:")
        print(f"  MSE: {metrics['mse']:.4f} | Pearson r: {metrics['pearson_r']:.4f} | CCC: {metrics['ccc']:.4f}")
        avg_mse.append(metrics['mse'])
        avg_r.append(metrics['pearson_r'])
        avg_ccc.append(metrics['ccc'])

    print("\n--- Average Metrics Across All Folds ---")
    print(f"Average MSE:       {np.nanmean(avg_mse):.4f}")
    print(f"Average Pearson r: {np.nanmean(avg_r):.4f}")
    print(f"Average CCC:       {np.nanmean(avg_ccc):.4f}")

# --- 5. Run the main function ---
# This will execute everything when you run the Jupyter cell
run_loocv()

Using device: cuda
Loaded existing 'valence_mean' variable.
Found 8 videos. Starting LOOCV...

LOOCV FOLD 1/8: Testing on 032SoloC_contextOnly.mp4
Initializing Training Dataset...
Scanning 7 video file(s) and building frame map...
Done. Found 12288 total frames for this dataset.
Initializing Test Dataset...
Scanning 1 video file(s) and building frame map...
Done. Found 1500 total frames for this dataset.
--- Starting Training for fold 1 ---


Fold 1 | Epoch 1/10 --- Average Train Loss: 0.263088


Fold 1 | Epoch 2/10 --- Average Train Loss: 0.090034


Fold 1 | Epoch 3/10 --- Average Train Loss: 0.042121


Fold 1 | Epoch 4/10 --- Average Train Loss: 0.025653


...

Fold 4 | Epoch 8/10 --- Average Train Loss: 0.012507


Fold 4 | Epoch 9/10 --- Average Train Loss: 0.010410


Fold 4 | Epoch 10/10 --- Average Train Loss: 0.008560
--- Training Finished for fold 4 ---
--- Evaluating on Exp2_011_context-only.mp4 ---



--- Results for Exp2_011_context-only.mp4 ---
  Mean Squared Error (MSE): 0.8984
  Pearson Correlation (r):  -0.2690
  Concordance Corr (CCC):   -0.1788


LOOCV FOLD 5/8: Testing on 031SoloC_contextOnly.mp4
Initializing Training Dataset...
Scanning 7 video file(s) and building frame map...
Done. Found 12288 total frames for this dataset.
Initializing Test Dataset...
Scanning 1 video file(s) and building frame map...
Done. Found 1500 total frames for this dataset.
--- Starting Training for fold 5 ---


Epoch 1/10:  42%|████▏     | 40/96 [30:20<35:18, 37.83s/it, loss=0.3762]  

In [34]:
avg_r

NameError: name 'avg_r' is not defined